# Part I · Block 1 — Your First Single-Block Run
**Tasks 1 & 2** | *Worksheet §Part I, Block 1 (25 min)*

This notebook runs the single-block propagation script on **Experiment 1, Site 1 (WT, ERK signal)**,
then guides you through inspecting the output tables.

Refer to the companion handout for all mathematical notation ($V$, $E_\mathrm{sp}$, $\theta$, $\mathrm{RR}$, etc.).

---
## Setup

In [ ]:
import sys, json, subprocess
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.ticker as ticker

# ── project layout ────────────────────────────────────────────────────────────
PROJECT_ROOT = Path('..').resolve()
SCRIPTS_DIR  = PROJECT_ROOT / 'scripts'
DATA_PATH    = PROJECT_ROOT / 'single-cell-tracks_exp1-6_noErbB2.csv.gz'
META_PATH    = PROJECT_ROOT / '01-readme-experiment-description_2022-04-05.csv'
OUTPUT_ROOT  = PROJECT_ROOT / 'analysis_outputs'

# ── analysis parameters (Task 1 defaults) ─────────────────────────────────────
EXP_ID        = 1
SITE_ID       = 1
SIGNAL_COL    = 'ERKKTR_ratio'
SPATIAL_RADIUS = 60
FUTURE_WINDOW  = 3
JUMP_QUANTILE  = 0.9

# ── helper: path to outputs for a given run ───────────────────────────────────
def output_dir(exp, site, signal=SIGNAL_COL):
    return OUTPUT_ROOT / f'exp_{exp}_site_{site}_{signal}'

def load_summary(exp, site, signal=SIGNAL_COL):
    path = output_dir(exp, site, signal) / 'summary.json'
    with open(path) as f:
        return json.load(f)

print('Project root:', PROJECT_ROOT)
print('Data file exists:', DATA_PATH.exists())

---
## Task 1 — Setup and first run

Run the single-block analysis script via the command below.
The cell captures stdout and parses the `summary.json` output automatically.

In [ ]:
cmd = [
    sys.executable,
    str(SCRIPTS_DIR / 'spatiotemporal_signal_propagation.py'),
    '--data-path',           str(DATA_PATH),
    '--meta-path',           str(META_PATH),
    '--exp-id',              str(EXP_ID),
    '--site-id',             str(SITE_ID),
    '--signal-col',          SIGNAL_COL,
    '--spatial-radius',      str(SPATIAL_RADIUS),
    '--future-window-frames', str(FUTURE_WINDOW),
    '--jump-quantile',       str(JUMP_QUANTILE),
    '--output-dir',          str(OUTPUT_ROOT),
]

print('Running script …')
result = subprocess.run(cmd, capture_output=True, text=True)
print(result.stdout)
if result.returncode != 0:
    print('STDERR:', result.stderr)

In [ ]:
# Load the structured summary and print key quantities
s = load_summary(EXP_ID, SITE_ID)

print(f"{'Nodes |V|':<40} {s['n_nodes']:>10,}")
print(f"{'Spatial edges |E_sp|':<40} {s['n_spatial_edges']:>10,}")
print(f"{'Temporal edges |E_tm|':<40} {s['n_temporal_edges']:>10,}")
print(f"{'Tracks':<40} {s['n_tracks']:>10,}")
print(f"{'Frames':<40} {s['n_frames']:>10,}")
print(f"{'Jump threshold θ':<40} {s['jump_threshold']:>10.4f}")
print(f"{'p_exp  (future jump | exposed)':<40} {s['future_jump_rate_if_neighbor_jumps_now']:>10.4f}")
print(f"{'p_unexp (future jump | unexposed)':<40} {s['future_jump_rate_if_no_neighbor_jumps_now']:>10.4f}")
print(f"{'Risk difference RD':<40} {s['risk_difference']:>10.4f}")
print(f"{'Relative risk RR':<40} {s['relative_risk']:>10.4f}")

### Task 1 — Questions

**(a)** Compute the ratio $|E_\mathrm{sp}|/|V|$. What does it tell you about local cell density?

In [ ]:
ratio = s['n_spatial_edges'] / s['n_nodes']
print(f"|E_sp| / |V| = {ratio:.2f}")

# ── YOUR INTERPRETATION ──────────────────────────────────────────────────────
# Write a brief comment here explaining what this ratio means for cell density.


**(b)** What is the data-driven threshold $\theta_{0.90}$? In your own words, what does crossing this value mean for a cell at node $(k,t)$?

In [ ]:
print(f"θ_0.90 = {s['jump_threshold']:.4f}")

# ── YOUR INTERPRETATION ──────────────────────────────────────────────────────


**(c)** Is $\mathrm{RR} > 1$, $= 1$, or $< 1$? State in one biological sentence what this implies.

In [ ]:
print(f"RR = {s['relative_risk']:.4f}  →  {'> 1 (positive association)' if s['relative_risk'] > 1 else '≤ 1'}")

# ── YOUR BIOLOGICAL INTERPRETATION ───────────────────────────────────────────


---
## Task 2 — Inspect the output tables

Load the node table produced by the script.

In [ ]:
nodes = pd.read_csv(output_dir(EXP_ID, SITE_ID) / 'nodes.csv.gz')
print(f"Shape: {nodes.shape}")
nodes.head(3)

### Task 2(a) — Plot signal traces for three tracks

For three tracks of your choice, plot $S_k(t)$ over time and mark jump events ($\mathcal{J}_k(t)=1$) as dots.

In [ ]:
# ── pick three tracks ─────────────────────────────────────────────────────────
track_ids = nodes['track_id'].value_counts().index[:3].tolist()  # longest tracks

fig, axes = plt.subplots(1, 3, figsize=(14, 3), sharey=True)
for ax, tid in zip(axes, track_ids):
    tr = nodes[nodes['track_id'] == tid].sort_values('Image_Metadata_T')
    ax.plot(tr['time_h'], tr['signal_value'], lw=1.2, color='steelblue', label='Signal')

    # ── YOUR CODE: mark jump events ───────────────────────────────────────────
    jumps = tr[tr['jump_event'] == True]
    # ax.scatter(...)   ← add scatter call for jump frames

    ax.set_title(f'Track {tid}', fontsize=9)
    ax.set_xlabel('Time (h)')

axes[0].set_ylabel(SIGNAL_COL)
plt.tight_layout()
plt.show()

# ── YOUR OBSERVATION ─────────────────────────────────────────────────────────
# Are jumps isolated frames or do they appear in bursts?


### Task 2(b) — Verify $p_\mathrm{exp}$ by filtering

In [ ]:
exposed = nodes[nodes['neighbor_jump_now'] == True]

# ── YOUR CODE ────────────────────────────────────────────────────────────────
# Compute the fraction of exposed nodes where future_self_jump == True
# and verify it matches s['future_jump_rate_if_neighbor_jumps_now']

p_exp_manual = exposed['future_self_jump'].mean()
print(f"p_exp from filter : {p_exp_manual:.4f}")
print(f"p_exp from summary: {s['future_jump_rate_if_neighbor_jumps_now']:.4f}")
print(f"Match: {abs(p_exp_manual - s['future_jump_rate_if_neighbor_jumps_now']) < 1e-6}")

### Task 2(c) — Distribution of neighbour count $n(k,t)$

In [ ]:
nc = nodes['neighbor_count']
print(f"mean  n(k,t) = {nc.mean():.2f}")
print(f"std   n(k,t) = {nc.std():.2f}")
print(f"max   n(k,t) = {nc.max()}")

fig, ax = plt.subplots(figsize=(6, 3))

# ── YOUR CODE ────────────────────────────────────────────────────────────────
# Plot a histogram of neighbor_count.
# ax.hist(...)

ax.set_xlabel('n(k, t)  — spatial neighbour count')
ax.set_ylabel('Number of nodes')
ax.set_title(f'Neighbour count distribution  (r = {SPATIAL_RADIUS})')
plt.tight_layout()
plt.show()

# ── YOUR INTERPRETATION ──────────────────────────────────────────────────────
# Is the neighbourhood sparse or dense at r = 60?
